In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [16]:
from transformers import T5Config, AutoTokenizer, T5Tokenizer
from emotion_embed_T5_class import T5WithEmotionEmbeddings 

tokenizer_embed = T5Tokenizer.from_pretrained(
    "./t5-emotion-embed-finetuned/tokenizer",
    add_prefix_space=False  
)

config_embed = T5Config.from_pretrained("./t5-emotion-embed-finetuned/model")

model_embed = T5WithEmotionEmbeddings.from_pretrained(
    "./t5-emotion-embed-finetuned/model",
    config=config_embed
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_embed.to(device)

T5WithEmotionEmbeddings(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Dropout

In [18]:
from emotion_classifier import classify_emotion
from transformers import AutoTokenizer
import torch


plutchik_emotion2id = {
    'trust': 0,
    'joy': 1,
    'anger': 2,
    'anticipation': 3,
    'fear': 4,
    'sadness': 5,
    'disgust': 6,
    'surprise': 7,
    'neutral': 8
}

def generate_response(input_text):
    model_embed.to(device)

    # Classify the emotion
    emotion = classify_emotion(input_text)
    emotion_id = torch.tensor([plutchik_emotion2id.get(emotion, 0)], device=device)  # Default to 'neutral' if unknown

    # Tokenize input
    inputs = tokenizer_embed(input_text, return_tensors="pt", truncation=True, padding=True).to(device)

    # Generate with emotion embedding
    try:
        outputs = model_embed.generate(
            **inputs,
            emotion_ids=emotion_id,
            max_length=50,
            do_sample=True,
            top_p=0.95,
            temperature=0.7
        )
        responses = [tokenizer_embed.decode(outputs[0], skip_special_tokens=True)]
    except TypeError:
        # If model doesn't accept emotion_ids directly in generate (shouldn't happen now), try sampling multiple
        outputs = model_embed.generate(
            **inputs,
            emotion_ids=emotion_id,
            max_length=50,
            do_sample=True,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=5
        )
        responses = [tokenizer_embed.decode(o, skip_special_tokens=True) for o in outputs]

    # Evaluate all responses and pick the best using F1 score
    P, R, F1 = score(responses, [input_text]*len(responses), lang='en', verbose=False)
    best_response = responses[F1.argmax()]

    return best_response


In [19]:
print(generate_response("I lost my pet"))  

ValueError: You have to specify either decoder_input_ids or decoder_inputs_embeds

In [41]:
emotion = classify_emotion("I lost my pet")
print(f"Detected emotion: {emotion}")

Detected emotion: sadness


In [47]:
print(generate_response("We won the championship!"))

oh wow thats great!
